In [0]:
%sql
CREATE TABLE IF NOT EXISTS control_table (
    config_id BIGINT,
    source_name STRING,
    source_type STRING,
    source_entity STRING,
    source_format STRING,
    target_layer STRING,
    target_table STRING,
    load_type STRING,
    is_streaming BOOLEAN,
    trigger_interval_seconds INT,
    watermark_column STRING,
    max_retries INT,
    retry_interval_seconds INT,
    is_active BOOLEAN,
    environment STRING,
    created_at TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA;

In [0]:
SHOW TABLES;

In [0]:
INSERT INTO control_table VALUES
(1, 'Binance', 'WebSocket', 'BTCUSDT', 'JSON',
 'bronze', 'bronze_crypto_trades', 'streaming', true,
 10, 'event_timestamp', 3, 30, true, 'DEV',
 current_timestamp(), current_timestamp()),

(2, 'Binance', 'WebSocket', 'ETHUSDT', 'JSON',
 'bronze', 'bronze_crypto_trades', 'streaming', true,
 10, 'event_timestamp', 3, 30, true, 'DEV',
 current_timestamp(), current_timestamp()),

(3, 'CoinMasterAPI', 'REST', 'coins', 'JSON',
 'bronze', 'bronze_coin_master', 'incremental', false,
 86400, 'updated_at', 3, 60, true, 'DEV',
 current_timestamp(), current_timestamp()),

(4, 'ExchangeMasterAPI', 'REST', 'exchanges', 'JSON',
 'bronze', 'bronze_exchange_master', 'incremental', false,
 86400, 'updated_at', 3, 60, true, 'DEV',
 current_timestamp(), current_timestamp());

In [0]:
SELECT *
FROM control_table
ORDER BY config_id;

In [0]:
%python
import requests

url = "https://api.binance.com/api/v3/ping"

response = requests.get(url, timeout=10)

print(response.status_code)
print(response.text)

In [0]:
%python
import requests

url = "https://api.kraken.com/0/public/Ticker?pair=XBTUSD"

response = requests.get(url, timeout=10)

print(response.status_code)
print(response.text[:500])

In [0]:
UPDATE control_table
SET
    source_name = 'Kraken',
    source_type = 'WebSocket',
    source_entity = 'BTC/USD',
    source_format = 'JSON',
    target_layer = 'bronze',
    target_table = 'bronze_crypto_trades',
    load_type = 'streaming',
    is_streaming = true,
    trigger_interval_seconds = 10,
    watermark_column = 'event_timestamp',
    max_retries = 3,
    retry_interval_seconds = 30,
    updated_at = current_timestamp()
WHERE config_id = 1;

In [0]:
SELECT *
FROM control_table
ORDER BY config_id;

In [0]:
ALTER TABLE control_table
ADD COLUMNS (
    source_endpoint STRING
);

In [0]:
UPDATE control_table
SET source_endpoint = 'wss://ws.kraken.com/v2'
WHERE config_id = 1;

In [0]:
SELECT
    config_id,
    source_name,
    source_type,
    source_entity,
    source_endpoint,
    target_table
FROM control_table
WHERE config_id = 1;